<a href="https://colab.research.google.com/github/Subroy1/MSAI_AllPracticeModules/blob/main/Module%2015/MultiDimensional_Gradient_Descent_Tips_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
#tips dataset
tips=sns.load_dataset("tips")
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


Imrove the moddel so that we want to predict the tip from the total_bill plus a constant offset .
tip = theta0 + theta1*bill

In [3]:
model = LinearRegression(fit_intercept=True)
X=tips[["total_bill"]]
y=tips["tip"]
model.fit(X,y)

LinearRegression()

In [4]:
model.intercept_, model.coef_

(np.float64(0.9202696135546731), array([0.10502452]))

This exercise is to compute the coef and intercept from scratch without using library.

In [5]:
tips_with_bias= tips.copy()
tips_with_bias["bias"]= 1
X=tips_with_bias[["bias","total_bill"]]
X.head()

,bias,total_bill
0,1,16.99
1,1,10.34
2,1,21.01
3,1,23.68
4,1,24.59


In [6]:
model = LinearRegression(fit_intercept=False)
model.fit(X,y)
model.coef_

array([0.92026961, 0.10502452])

In [7]:
y_preds  = X@np.array([model.coef_[0],model.coef_[1]])
y_preds

,0
0,2.704636
1,2.006223
2,3.126835
3,3.407250
4,3.502822
...,...
239,3.969131
240,3.774836
241,3.301175
242,2.791807


In [8]:
def mse_loss (theta, X,y):
    y_hat=theta[0]*X.iloc[:,0] + theta[1]*X.iloc[:,1]
    return np.mean((y_hat-y)**2)

In [9]:
mse_loss(np.array([model.coef_[0],model.coef_[1]]), X,y)

np.float64(1.036019442011377)

In [10]:
def mse_loss_single_arg(theta):
    return mse_loss(theta, X,y)

In [11]:
mse_loss_single_arg(np.array([model.coef_[0],model.coef_[1]]))

np.float64(1.036019442011377)

In [13]:
mse_loss_single_arg([1.5,0.0])

np.float64(4.1514475409836065)

In [15]:
# Using this function, create a 3D plot uising plotly_graph_object
import plotly.graph_objects as go
import numpy as np

# Define the range for theta0 and theta1
theta0_range = np.linspace(-5, 5, 100)  # Intercept
theta1_range = np.linspace(-1, 1, 100)  # Slope

# Create a meshgrid for theta0 and theta1
T0, T1 = np.meshgrid(theta0_range, theta1_range)

# Calculate the MSE loss for each combination of theta0 and theta1
Z = np.array([mse_loss_single_arg([t0, t1]) for t0, t1 in zip(T0.ravel(), T1.ravel())])
Z = Z.reshape(T0.shape)

# Find the index of the minimum MSE
min_mse_idx = np.unravel_index(np.argmin(Z, axis=None), Z.shape)
min_theta0 = T0[min_mse_idx]
min_theta1 = T1[min_mse_idx]
min_mse = Z[min_mse_idx]

# Create the 3D surface plot
fig = go.Figure(data=[go.Surface(z=Z, x=T0, y=T1)])

# Add a red marker for the lowest MSE point
fig.add_trace(go.Scatter3d(
    x=[min_theta0],
    y=[min_theta1],
    z=[min_mse],
    mode='markers',
    marker=dict(
        size=10,
        color='red',
        symbol='circle'
    ),
    name=f'Lowest MSE (theta0={min_theta0:.2f}, theta1={min_theta1:.2f}, MSE={min_mse:.2f})'
))

fig.update_layout(
    title='MSE Loss Surface with Lowest MSE Highlighted',
    scene=dict(
        xaxis_title='Theta0 (Intercept)',
        yaxis_title='Theta1 (Slope)',
        zaxis_title='MSE Loss'
    )
)

fig.show()

With scipy.optimize minimize ## Uses gradient descent

In [16]:
import scipy
scipy.optimize.minimize(mse_loss_single_arg, x0=[0,0])

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 1.0360194420114932
        x: [ 9.203e-01  1.050e-01]
      nit: 3
      jac: [-4.470e-08 -2.980e-08]
 hess_inv: [[ 2.980e+00 -1.253e-01]
            [-1.253e-01  6.335e-03]]
     nfev: 15
     njev: 5